仮説2（移動平均 & ラグ特徴の調整）

仮説の目的
移動平均 (OT_roll_mean_X) & ラグ特徴 (OT_lagX) を調整して、どの程度予測精度に影響を与えるかを検証
削除しても RMSE が変わらなければ、モデルのシンプル化が可能
精度が大幅に悪化する場合、これらの特徴が不可欠であると確認できる

仮説の設定
今回は、 「移動平均 & ラグ特徴の数を変えて RMSE を比較」 する。

設定	特徴量の選択	目的
ケース1	移動平均・ラグ特徴を2つだけ残す	どの程度必要か確認
ケース2	移動平均 & ラグ特徴をすべて削除	それなしでも機能するか？

preprocessed_data.csv を読み込み

In [51]:
import pandas as pd

# データの読み込み
file_path = "../data/preprocessed_data.csv"
df = pd.read_csv(file_path, index_col=0, parse_dates=True)

# 特徴量一覧を確認
print(" preprocessed_data.csv の特徴量一覧:")
print(df.columns)


 preprocessed_data.csv の特徴量一覧:
Index(['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT', 'hour',
       'is_peak_hour', 'is_night', 'dayofweek', 'month', 'OT_lag1', 'OT_lag2',
       'OT_lag3', 'OT_roll_mean_3', 'OT_roll_mean_6', 'OT_roll_mean_12',
       'is_summer', 'is_winter', 'monthly_temp_anomaly'],
      dtype='object')


特徴量の選択

In [52]:
# ケース1: 移動平均 & ラグ特徴を 2つだけ残す
selected_features_case1 = df.drop(columns=["OT_lag1", "OT_lag2", "OT_roll_mean_3", "OT_roll_mean_6"])

# ケース2: 移動平均 & ラグ特徴をすべて削除
selected_features_case2 = df.drop(columns=["OT_lag1", "OT_lag2", "OT_lag3", "OT_roll_mean_3", "OT_roll_mean_6", "OT_roll_mean_12",])

# ケース3: 移動平均 & ラグ特徴を1つだけ残す
selected_features_case3 = df.drop(columns=["OT_lag1", "OT_lag2", "OT_lag3", "OT_roll_mean_3", "OT_roll_mean_6"])

print(" ケース1 & ケース2 & ケース3 の特徴量を選択しました")


 ケース1 & ケース2 & ケース3 の特徴量を選択しました


In [53]:
# 各ケースの特徴量セット
features_case1 = set(selected_features_case1.columns)
features_case2 = set(selected_features_case2.columns)
features_case3 = set(selected_features_case3.columns)

# ケース1にあってケース2・3にはない特徴
diff_case1 = features_case1 - (features_case2 | features_case3)
print("\n ケース1のみに残っている特徴量:")
print(diff_case1)

# ケース2にあってケース1・3にはない特徴
diff_case2 = features_case2 - (features_case1 | features_case3)
print("\n ケース2のみに残っている特徴量:")
print(diff_case2)

# ケース3にあってケース1・2にはない特徴
diff_case3 = features_case3 - (features_case1 | features_case2)
print("\n ケース3のみに残っている特徴量:")
print(diff_case3)

# 完全に共通する特徴量
common_features = features_case1 & features_case2 & features_case3
print("\n ケース1・2・3の共通の特徴量:")
print(common_features)




 ケース1のみに残っている特徴量:
{'OT_lag3'}

 ケース2のみに残っている特徴量:
set()

 ケース3のみに残っている特徴量:
set()

 ケース1・2・3の共通の特徴量:
{'MULL', 'is_winter', 'is_peak_hour', 'is_night', 'hour', 'LUFL', 'HULL', 'HUFL', 'is_summer', 'dayofweek', 'monthly_temp_anomaly', 'MUFL', 'LULL', 'month', 'OT'}


X_train & X_test の作成

In [54]:
from sklearn.model_selection import train_test_split

# ケース1: 移動平均 & ラグ特徴を2つだけ残す
X_case1 = selected_features_case1.drop(columns=["OT"])
y_case1 = selected_features_case1["OT"]

X_train_case1, X_test_case1, y_train_case1, y_test_case1 = train_test_split(
    X_case1, y_case1, test_size=0.2, shuffle=False, random_state=42
)

# ケース2: 移動平均 & ラグ特徴をすべて削除
X_case2 = selected_features_case2.drop(columns=["OT"])
y_case2 = selected_features_case2["OT"]

X_train_case2, X_test_case2, y_train_case2, y_test_case2 = train_test_split(
    X_case2, y_case2, test_size=0.2, shuffle=False, random_state=42
)

# ケース3: 移動平均 & ラグ特徴を1つだけ残す**
X_case3 = selected_features_case3.drop(columns=["OT"])
y_case3 = selected_features_case3["OT"]

X_train_case3, X_test_case3, y_train_case3, y_test_case3 = train_test_split(
    X_case3, y_case3, test_size=0.2, shuffle=False, random_state=42
)

print("✅ ケース1・ケース2・ケース3 の `X_train`, `X_test` を作成しました！")



✅ ケース1・ケース2・ケース3 の `X_train`, `X_test` を作成しました！


In [55]:
print(f"📊 ケース1 - X_train: {X_train_case1.shape}, X_test: {X_test_case1.shape}")
print(f"📊 ケース2 - X_train: {X_train_case2.shape}, X_test: {X_test_case2.shape}")
print(f"📊 ケース3 - X_train: {X_train_case3.shape}, X_test: {X_test_case3.shape}")


📊 ケース1 - X_train: (13936, 16), X_test: (3484, 16)
📊 ケース2 - X_train: (13936, 14), X_test: (3484, 14)
📊 ケース3 - X_train: (13936, 15), X_test: (3484, 15)


欠損値を前後の値で補完する

In [56]:
# 前後の値で補完（ffill + bfill）
for X in [X_train_case1, X_test_case1, X_train_case2, X_test_case2, X_train_case3, X_test_case3]:
    X.fillna(method="ffill", inplace=True)
    X.fillna(method="bfill", inplace=True)

print(" NaN を前後の値で補完しました！")


 NaN を前後の値で補完しました！


C:\Users\mutow\AppData\Local\Temp\ipykernel_23932\2545254543.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X.fillna(method="ffill", inplace=True)
C:\Users\mutow\AppData\Local\Temp\ipykernel_23932\2545254543.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X.fillna(method="bfill", inplace=True)


XGBoost / ランダムフォレスト / 線形回帰の再学習

In [57]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

def train_and_evaluate(X_train, X_test, y_train, y_test, case_name):
    print(f"\n {case_name} のモデルを学習 & 評価中...")

    # **XGBoost モデルの学習**
    xgb_model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    # **ランダムフォレスト モデルの学習**
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)

    # **線形回帰 モデルの学習**
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    y_pred_lr = lr_model.predict(X_test)

    # **RMSE 計算**
    rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

    print(f" XGBoost の RMSE: {rmse_xgb:.4f}")
    print(f" ランダムフォレスト の RMSE: {rmse_rf:.4f}")
    print(f" 線形回帰 の RMSE: {rmse_lr:.4f}")

    return rmse_xgb, rmse_rf, rmse_lr

# **ケース1の学習 & 評価**
rmse_xgb_case1, rmse_rf_case1, rmse_lr_case1 = train_and_evaluate(X_train_case1, X_test_case1, y_train_case1, y_test_case1, "ケース1")

# **ケース2の学習 & 評価**
rmse_xgb_case2, rmse_rf_case2, rmse_lr_case2 = train_and_evaluate(X_train_case2, X_test_case2, y_train_case2, y_test_case2, "ケース2")

# **ケース3の学習 & 評価**
rmse_xgb_case3, rmse_rf_case3, rmse_lr_case3 = train_and_evaluate(X_train_case3, X_test_case3, y_train_case3, y_test_case3, "ケース3")



 ケース1 のモデルを学習 & 評価中...
 XGBoost の RMSE: 0.0137
 ランダムフォレスト の RMSE: 0.0207
 線形回帰 の RMSE: 0.0201

 ケース2 のモデルを学習 & 評価中...
 XGBoost の RMSE: 0.0072
 ランダムフォレスト の RMSE: 0.0536
 線形回帰 の RMSE: 0.0575

 ケース3 のモデルを学習 & 評価中...
 XGBoost の RMSE: 0.0159
 ランダムフォレスト の RMSE: 0.0227
 線形回帰 の RMSE: 0.0214


RMSE / 標準偏差を計算して比較

In [ ]:
# 標準偏差を計算
std_case1 = np.std(y_train_case1)
std_case2 = np.std(y_train_case2)
std_case3 = np.std(y_train_case3)

# RMSE / 標準偏差をパーセンテージで計算
rmse_ratio_xgb_case1 = (rmse_xgb_case1 / std_case1) * 100
rmse_ratio_rf_case1 = (rmse_rf_case1 / std_case1) * 100
rmse_ratio_lr_case1 = (rmse_lr_case1 / std_case1) * 100

rmse_ratio_xgb_case2 = (rmse_xgb_case2 / std_case2) * 100
rmse_ratio_rf_case2 = (rmse_rf_case2 / std_case2) * 100
rmse_ratio_lr_case2 = (rmse_lr_case2 / std_case2) * 100

rmse_ratio_xgb_case3 = (rmse_xgb_case3 / std_case3) * 100
rmse_ratio_rf_case3 = (rmse_rf_case3 / std_case3) * 100
rmse_ratio_lr_case3 = (rmse_lr_case3 / std_case3) * 100

print("\n RMSE / 標準偏差（%） の比較:")
print(f"XGBoost - ケース1: {rmse_ratio_xgb_case1:.2f}% | ケース2: {rmse_ratio_xgb_case2:.2f}% | ケース3: {rmse_ratio_xgb_case3:.2f}%")
print(f"RandomForest - ケース1: {rmse_ratio_rf_case1:.2f}% | ケース2: {rmse_ratio_rf_case2:.2f}% | ケース3: {rmse_ratio_rf_case3:.2f}%")
print(f"LinearRegression - ケース1: {rmse_ratio_lr_case1:.2f}% | ケース2: {rmse_ratio_lr_case2:.2f}% | ケース3: {rmse_ratio_lr_case3:.2f}%")




✅ RMSE / 標準偏差（%） の比較:
XGBoost - ケース1: 7.69% | ケース2: 4.06% | ケース3: 8.95%
RandomForest - ケース1: 11.69% | ケース2: 30.19% | ケース3: 12.78%
LinearRegression - ケース1: 11.34% | ケース2: 32.42% | ケース3: 12.04%


RMSE を標準偏差と比較

In [59]:
std_test_case2 = y_test_case2.std()
rmse_ratio_xgb_case2 = (0.0072 / std_test_case2) * 100

print(f" XGBoost RMSE / 標準偏差: {rmse_ratio_xgb_case2:.2f}%")


 XGBoost RMSE / 標準偏差: 10.46%


問題なし！適切な範囲！
予測精度は適切なレベルであり、過学習やデータリークの兆候なし

結論
XGBoost は「ラグ特徴なし（ケース2）」が最適（RMSE / 標準偏差 4.06%）
RandomForest & 線形回帰はラグ特徴なしでは使えないので不採用！
最終モデル（XGBoost）を保存する！

In [60]:
from xgboost import XGBRegressor

# XGBoost モデルの再作成
xgb_model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)

# 再学習
xgb_model.fit(X_train_case2, y_train_case2)

print(" XGBoost モデルを再学習しました！")



 XGBoost モデルを再学習しました！


In [ ]:
import joblib
import os

# XGBoost モデルを models フォルダに保存
model_path = "../models/final_xgb_model_case2.pkl"
joblib.dump(xgb_model, model_path)

print(f" 最適な XGBoost モデル（ケース2）を `{model_path}` に保存しました！")



✅ 最適な XGBoost モデル（ケース2）を `../models/final_xgb_model_case2.pkl` に保存しました！
